In [ ]:
# Install Ultralytics — the library that contains YOLOv8
# YOLOv8 is the object detection model we use to detect traffic signs
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.5 MB/s eta 0:00:00


In [ ]:
# ==============================
# 2. MOUNT GOOGLE DRIVE
# ==============================

from google.colab import drive
drive.mount('/content/drive')
data_preparation.ipynb
data_preparation.ipynb_

[ ]
# Install Ultralytics — the library that contains YOLOv8
# YOLOv8 is the object detection model we use to detect traffic signs
!pip install ultralytics -q
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.5 MB/s eta 0:00:00

[ ]
# ==============================
# 2. MOUNT GOOGLE DRIVE
# ==============================

from google.colab import drive
drive.mount('/content/drive')
Mounted at /content/drive

[ ]
import os, shutil, random
from PIL import Image

# Path to your dataset folder (change if needed)
DATASET_PATH = "/content/drive/MyDrive/research/TrainIJCNN2013"

# Ground truth file (contains bounding boxes + class IDs)
GT_FILE = os.path.join(DATASET_PATH, "gt.txt")

# Where YOLO datasets will be created

Dataset exists: True
gt.txt exists: True

[ ]
# ==============================
# 3. DEFINE CLASSES
# ==============================

# Vehicle A - Prohibitory signs
prohibitory = [0,1,2,3,4,5,7,8,9,10,15,16]

# Vehicle B - Danger signs
danger = [11,18,19,20,21,22,23,24,25,26,27,28,29,30,31]

# Vehicle C - Mandatory signs
mandatory = [33,34,35,36,37,38,39,40]

# Global model - ALL 43 classes, including "other"
all_classes = list(range(43))

datasets = {
    "vehicle_A_prohibitory": prohibitory,
    "vehicle_B_danger": danger,
    "vehicle_C_mandatory": mandatory,
    "global_all": all_classes
}

[ ]
# Convert bounding box from (left, top, right, bottom)
# to YOLO format (x_center, y_center, width, height)
def convert_bbox_to_yolo(left, top, right, bottom, img_w, img_h):
    x_center = ((left + right) / 2) / img_w
    y_center = ((top + bottom) / 2) / img_h
    width = (right - left) / img_w
    height = (bottom - top) / img_h
    return x_center, y_center, width, height

[ ]
# ==============================
# CELL 6: CREATE YOLO DATASET FUNCTION
# ==============================

def create_yolo_dataset(dataset_name, selected_classes):
    """
    Creates YOLO dataset for one model:
    - Vehicle A: prohibitory only
    - Vehicle B: danger only
    - Vehicle C: mandatory only
    - Global: all 43 classes
    """

    output_dir = os.path.join(OUTPUT_BASE, dataset_name)

    # Remove old dataset to avoid old .ppm files or wrong labels
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)

    # Create folder structure
    os.makedirs(os.path.join(output_dir, "images", "train"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "images", "val"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "labels", "train"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "labels", "val"), exist_ok=True)

    # YOLO class IDs must be continuous starting from 0
    # Example: original class 33 becomes 0 in Vehicle C
    class_map = {old_id: new_id for new_id, old_id in enumerate(selected_classes)}

    annotations = {}

    # Read gt.txt
    with open(GT_FILE, "r") as f:
        for line in f:
            parts = line.strip().split(";")

            # Expected: image;left;top;right;bottom;class_id
            if len(parts) != 6:
                continue

            img_name = parts[0].strip()
            left, top, right, bottom = map(int, parts[1:5])
            class_id = int(parts[5].strip())

            # Keep only classes needed for this dataset
            if class_id not in selected_classes:
                continue

            if img_name not in annotations:
                annotations[img_name] = []

            annotations[img_name].append((left, top, right, bottom, class_id))

    # Images with selected classes
    image_names = list(annotations.keys())
    random.shuffle(image_names)

    # 80% training, 20% validation
    split_index = int(len(image_names) * 0.8)
    train_images = image_names[:split_index]
    val_images = image_names[split_index:]

    def save_split(image_list, split):
        """
        Save images and YOLO labels for train or val.
        Converts .ppm to .jpg because YOLO in Colab may not detect .ppm properly.
        """

        image_dir = os.path.join(output_dir, "images", split)
        label_dir = os.path.join(output_dir, "labels", split)

        os.makedirs(image_dir, exist_ok=True)
        os.makedirs(label_dir, exist_ok=True)

        for img_name in image_list:
            img_path = os.path.join(DATASET_PATH, img_name)

            if not os.path.exists(img_path):
                print("Missing image:", img_path)
                continue

            # Open original .ppm image and convert to RGB
            img = Image.open(img_path).convert("RGB")
            img_w, img_h = img.size

            # Save as .jpg for YOLO compatibility
            new_img_name = img_name.replace(".ppm", ".jpg")
            image_save_path = os.path.join(image_dir, new_img_name)
            img.save(image_save_path, quality=95)

            # Label file must have same base name as image
            label_name = new_img_name.replace(".jpg", ".txt")
            label_path = os.path.join(label_dir, label_name)

            with open(label_path, "w") as label_file:
                for left, top, right, bottom, class_id in annotations[img_name]:
                    new_class_id = class_map[class_id]

                    x, y, w, h = convert_bbox_to_yolo(
                        left, top, right, bottom, img_w, img_h
                    )

                    label_file.write(f"{new_class_id} {x} {y} {w} {h}\n")

    # Save train and validation data
    save_split(train_images, "train")
    save_split(val_images, "val")

    # Create YAML file for YOLO training
    yaml_path = os.path.join(output_dir, f"{dataset_name}.yaml")

    yaml_content = f"""path: {output_dir}
train: images/train
val: images/val

names:
"""

    # Class names for now are class_originalID
    for new_id, old_id in enumerate(selected_classes):
        yaml_content += f"  {new_id}: class_{old_id}\n"

    with open(yaml_path, "w") as f:
        f.write(yaml_content)

    print("Created:", dataset_name)
    print("Classes:", len(selected_classes))
    print("Train images:", len(train_images))
    print("Val images:", len(val_images))
    print("Total images used:", len(image_names))
    print("YAML file:", yaml_path)
    print("------------------------------------------------")

[ ]
# ==============================
# 6. CREATE ALL DATASETS
# ==============================

for dataset_name, classes in datasets.items():
    create_yolo_dataset(dataset_name, classes)
Created: vehicle_A_prohibitory
Classes: 12
Train images: 211
Val images: 53
Total images used: 264
YAML file: /content/drive/MyDrive/research/yolo_datasets/vehicle_A_prohibitory/vehicle_A_prohibitory.yaml
------------------------------------------------
Created: vehicle_B_danger
Classes: 15
Train images: 100
Val images: 25
Total images used: 125
YAML file: /content/drive/MyDrive/research/yolo_datasets/vehicle_B_danger/vehicle_B_danger.yaml
------------------------------------------------
Created: vehicle_C_mandatory
Classes: 8
Train images: 80
Val images: 20
Total images used: 100
YAML file: /content/drive/MyDrive/research/yolo_datasets/vehicle_C_mandatory/vehicle_C_mandatory.yaml
------------------------------------------------
Created: global_all
Classes: 43
Train images: 404
Val images: 102
Total images used: 506
YAML file: /content/drive/MyDrive/research/yolo_datasets/global_all/global_all.yaml
------------------------------------------------

[ ]
# Check Vehicle A train images
!ls /content/drive/MyDrive/research/yolo_datasets/vehicle_A_prohibitory/images/train | head
00003.jpg
00004.jpg
00005.jpg
00007.jpg
00008.jpg
00009.jpg
00011.jpg
00012.jpg
00013.jpg
00018.jpg

[ ]

Start coding or generate with AI.

[ ]

Start coding or generate with AI.

[ ]
# ==============================
# CREATE GENERIC TRAFFIC SIGN DATASET
# ==============================
# This dataset is used only to detect:
# "There is a traffic sign"
#
# It does NOT classify sign category.
# All traffic signs become one class:
# class 0 = traffic_sign
# ==============================

def create_generic_traffic_sign_dataset():

    dataset_name = "generic_traffic_sign"

    output_dir = os.path.join(OUTPUT_BASE, dataset_name)

    # Remove old dataset if exists
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)

    # Create folder structure
    os.makedirs(os.path.join(output_dir, "images", "train"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "images", "val"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "labels", "train"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "labels", "val"), exist_ok=True)

    annotations = {}

    # Read gt.txt
    with open(GT_FILE, "r") as f:
        for line in f:

            parts = line.strip().split(";")

            if len(parts) != 6:
                continue

            img_name = parts[0].strip()

            left, top, right, bottom = map(int, parts[1:5])

            # Ignore original class ID
            # because ALL signs become class 0

            if img_name not in annotations:
                annotations[img_name] = []

            annotations[img_name].append(
                (left, top, right, bottom)
            )

    # Shuffle images
    image_names = list(annotations.keys())
    random.shuffle(image_names)

    # 80% train, 20% validation
    split_index = int(len(image_names) * 0.8)

    train_images = image_names[:split_index]
    val_images = image_names[split_index:]


    def save_split(image_list, split):

        image_dir = os.path.join(output_dir, "images", split)
        label_dir = os.path.join(output_dir, "labels", split)

        for img_name in image_list:

            img_path = os.path.join(DATASET_PATH, img_name)

            if not os.path.exists(img_path):
                continue

            # Open image
            img = Image.open(img_path).convert("RGB")

            img_w, img_h = img.size

            # Convert .ppm to .jpg
            new_img_name = img_name.replace(".ppm", ".jpg")

            image_save_path = os.path.join(
                image_dir,
                new_img_name
            )

            img.save(image_save_path, quality=95)

            # Create YOLO label file
            label_name = new_img_name.replace(".jpg", ".txt")

            label_path = os.path.join(
                label_dir,
                label_name
            )

            with open(label_path, "w") as label_file:

                for left, top, right, bottom in annotations[img_name]:

                    x, y, w, h = convert_bbox_to_yolo(
                        left,
                        top,
                        right,
                        bottom,
                        img_w,
                        img_h
                    )

                    # ALL SIGNS = CLASS 0
                    label_file.write(
                        f"0 {x} {y} {w} {h}\n"
                    )

    # Save datasets
    save_split(train_images, "train")
    save_split(val_images, "val")

    # Create YAML
    yaml_path = os.path.join(
        output_dir,
        f"{dataset_name}.yaml"
    )

    yaml_content = f"""path: {output_dir}
train: images/train
val: images/val

names:
  0: traffic_sign
"""

    with open(yaml_path, "w") as f:
        f.write(yaml_content)

    print("Created:", dataset_name)
    print("Classes: 1")
    print("Train images:", len(train_images))
    print("Val images:", len(val_images))
    print("Total images used:", len(image_names))
    print("YAML file:", yaml_path)

[ ]
import os
import shutil
import random
from PIL import Image

DATASET_PATH = "/content/drive/MyDrive/research/TrainIJCNN2013"

GT_FILE = os.path.join(DATASET_PATH, "gt.txt")

OUTPUT_BASE = "/content/drive/MyDrive/research/yolo_datasets"

[ ]
# ==============================
# CONVERT BBOX TO YOLO FORMAT
# ==============================

def convert_bbox_to_yolo(left, top, right, bottom, img_w, img_h):

    x_center = ((left + right) / 2) / img_w
    y_center = ((top + bottom) / 2) / img_h

    width = (right - left) / img_w
    height = (bottom - top) / img_h

    return x_center, y_center, width, height

[ ]
# ==============================
# CREATE GENERIC DATASET
# ==============================

create_generic_traffic_sign_dataset()
Created: generic_traffic_sign
Classes: 1
Train images: 404
Val images: 102
Total images used: 506
YAML file: /content/drive/MyDrive/research/yolo_datasets/generic_traffic_sign/generic_traffic_sign.yaml
Colab paid products - Cancel contracts here


Mounted at /content/drive


In [ ]:
import os, shutil, random
from PIL import Image

# Path to your dataset folder (change if needed)
DATASET_PATH = "/content/drive/MyDrive/research/TrainIJCNN2013"

# Ground truth file (contains bounding boxes + class IDs)
GT_FILE = os.path.join(DATASET_PATH, "gt.txt")

# Where YOLO datasets will be created
OUTPUT_BASE = "/content/drive/MyDrive/research/yolo_datasets"

# Check if paths exist
print("Dataset exists:", os.path.exists(DATASET_PATH))
print("gt.txt exists:", os.path.exists(GT_FILE))

Dataset exists: True
gt.txt exists: True


In [ ]:
# ==============================
# 3. DEFINE CLASSES
# ==============================

# Vehicle A - Prohibitory signs
prohibitory = [0,1,2,3,4,5,7,8,9,10,15,16]

# Vehicle B - Danger signs
danger = [11,18,19,20,21,22,23,24,25,26,27,28,29,30,31]

# Vehicle C - Mandatory signs
mandatory = [33,34,35,36,37,38,39,40]

# Global model - ALL 43 classes, including "other"
all_classes = list(range(43))

datasets = {
    "vehicle_A_prohibitory": prohibitory,
    "vehicle_B_danger": danger,
    "vehicle_C_mandatory": mandatory,
    "global_all": all_classes
}

In [ ]:
# Convert bounding box from (left, top, right, bottom)
# to YOLO format (x_center, y_center, width, height)
def convert_bbox_to_yolo(left, top, right, bottom, img_w, img_h):
    x_center = ((left + right) / 2) / img_w
    y_center = ((top + bottom) / 2) / img_h
    width = (right - left) / img_w
    height = (bottom - top) / img_h
    return x_center, y_center, width, height

In [ ]:
# ==============================
# CELL 6: CREATE YOLO DATASET FUNCTION
# ==============================

def create_yolo_dataset(dataset_name, selected_classes):
    """
    Creates YOLO dataset for one model:
    - Vehicle A: prohibitory only
    - Vehicle B: danger only
    - Vehicle C: mandatory only
    - Global: all 43 classes
    """

    output_dir = os.path.join(OUTPUT_BASE, dataset_name)

    # Remove old dataset to avoid old .ppm files or wrong labels
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)

    # Create folder structure
    os.makedirs(os.path.join(output_dir, "images", "train"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "images", "val"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "labels", "train"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "labels", "val"), exist_ok=True)

    # YOLO class IDs must be continuous starting from 0
    # Example: original class 33 becomes 0 in Vehicle C
    class_map = {old_id: new_id for new_id, old_id in enumerate(selected_classes)}

    annotations = {}

    # Read gt.txt
    with open(GT_FILE, "r") as f:
        for line in f:
            parts = line.strip().split(";")

            # Expected: image;left;top;right;bottom;class_id
            if len(parts) != 6:
                continue

            img_name = parts[0].strip()
            left, top, right, bottom = map(int, parts[1:5])
            class_id = int(parts[5].strip())

            # Keep only classes needed for this dataset
            if class_id not in selected_classes:
                continue

            if img_name not in annotations:
                annotations[img_name] = []

            annotations[img_name].append((left, top, right, bottom, class_id))

    # Images with selected classes
    image_names = list(annotations.keys())
    random.shuffle(image_names)

    # 80% training, 20% validation
    split_index = int(len(image_names) * 0.8)
    train_images = image_names[:split_index]
    val_images = image_names[split_index:]

    def save_split(image_list, split):
        """
        Save images and YOLO labels for train or val.
        Converts .ppm to .jpg because YOLO in Colab may not detect .ppm properly.
        """

        image_dir = os.path.join(output_dir, "images", split)
        label_dir = os.path.join(output_dir, "labels", split)

        os.makedirs(image_dir, exist_ok=True)
        os.makedirs(label_dir, exist_ok=True)

        for img_name in image_list:
            img_path = os.path.join(DATASET_PATH, img_name)

            if not os.path.exists(img_path):
                print("Missing image:", img_path)
                continue

            # Open original .ppm image and convert to RGB
            img = Image.open(img_path).convert("RGB")
            img_w, img_h = img.size

            # Save as .jpg for YOLO compatibility
            new_img_name = img_name.replace(".ppm", ".jpg")
            image_save_path = os.path.join(image_dir, new_img_name)
            img.save(image_save_path, quality=95)

            # Label file must have same base name as image
            label_name = new_img_name.replace(".jpg", ".txt")
            label_path = os.path.join(label_dir, label_name)

            with open(label_path, "w") as label_file:
                for left, top, right, bottom, class_id in annotations[img_name]:
                    new_class_id = class_map[class_id]

                    x, y, w, h = convert_bbox_to_yolo(
                        left, top, right, bottom, img_w, img_h
                    )

                    label_file.write(f"{new_class_id} {x} {y} {w} {h}\n")

    # Save train and validation data
    save_split(train_images, "train")
    save_split(val_images, "val")

    # Create YAML file for YOLO training
    yaml_path = os.path.join(output_dir, f"{dataset_name}.yaml")

    yaml_content = f"""path: {output_dir}
train: images/train
val: images/val

names:
"""

    # Class names for now are class_originalID
    for new_id, old_id in enumerate(selected_classes):
        yaml_content += f"  {new_id}: class_{old_id}\n"

    with open(yaml_path, "w") as f:
        f.write(yaml_content)

    print("Created:", dataset_name)
    print("Classes:", len(selected_classes))
    print("Train images:", len(train_images))
    print("Val images:", len(val_images))
    print("Total images used:", len(image_names))
    print("YAML file:", yaml_path)
    print("------------------------------------------------")

In [ ]:
# ==============================
# 6. CREATE ALL DATASETS
# ==============================

for dataset_name, classes in datasets.items():
    create_yolo_dataset(dataset_name, classes)

Created: vehicle_A_prohibitory
Classes: 12
Train images: 211
Val images: 53
Total images used: 264
YAML file: /content/drive/MyDrive/research/yolo_datasets/vehicle_A_prohibitory/vehicle_A_prohibitory.yaml
------------------------------------------------
Created: vehicle_B_danger
Classes: 15
Train images: 100
Val images: 25
Total images used: 125
YAML file: /content/drive/MyDrive/research/yolo_datasets/vehicle_B_danger/vehicle_B_danger.yaml
------------------------------------------------
Created: vehicle_C_mandatory
Classes: 8
Train images: 80
Val images: 20
Total images used: 100
YAML file: /content/drive/MyDrive/research/yolo_datasets/vehicle_C_mandatory/vehicle_C_mandatory.yaml
------------------------------------------------
Created: global_all
Classes: 43
Train images: 404
Val images: 102
Total images used: 506
YAML file: /content/drive/MyDrive/research/yolo_datasets/global_all/global_all.yaml
------------------------------------------------


In [ ]:
# Check Vehicle A train images
!ls /content/drive/MyDrive/research/yolo_datasets/vehicle_A_prohibitory/images/train | head

00003.jpg
00004.jpg
00005.jpg
00007.jpg
00008.jpg
00009.jpg
00011.jpg
00012.jpg
00013.jpg
00018.jpg


In [ ]:
# ==============================
# CREATE GENERIC TRAFFIC SIGN DATASET
# ==============================
# This dataset is used only to detect:
# "There is a traffic sign"
#
# It does NOT classify sign category.
# All traffic signs become one class:
# class 0 = traffic_sign
# ==============================

def create_generic_traffic_sign_dataset():

    dataset_name = "generic_traffic_sign"

    output_dir = os.path.join(OUTPUT_BASE, dataset_name)

    # Remove old dataset if exists
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)

    # Create folder structure
    os.makedirs(os.path.join(output_dir, "images", "train"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "images", "val"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "labels", "train"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "labels", "val"), exist_ok=True)

    annotations = {}

    # Read gt.txt
    with open(GT_FILE, "r") as f:
        for line in f:

            parts = line.strip().split(";")

            if len(parts) != 6:
                continue

            img_name = parts[0].strip()

            left, top, right, bottom = map(int, parts[1:5])

            # Ignore original class ID
            # because ALL signs become class 0

            if img_name not in annotations:
                annotations[img_name] = []

            annotations[img_name].append(
                (left, top, right, bottom)
            )

    # Shuffle images
    image_names = list(annotations.keys())
    random.shuffle(image_names)

    # 80% train, 20% validation
    split_index = int(len(image_names) * 0.8)

    train_images = image_names[:split_index]
    val_images = image_names[split_index:]


    def save_split(image_list, split):

        image_dir = os.path.join(output_dir, "images", split)
        label_dir = os.path.join(output_dir, "labels", split)

        for img_name in image_list:

            img_path = os.path.join(DATASET_PATH, img_name)

            if not os.path.exists(img_path):
                continue

            # Open image
            img = Image.open(img_path).convert("RGB")

            img_w, img_h = img.size

            # Convert .ppm to .jpg
            new_img_name = img_name.replace(".ppm", ".jpg")

            image_save_path = os.path.join(
                image_dir,
                new_img_name
            )

            img.save(image_save_path, quality=95)

            # Create YOLO label file
            label_name = new_img_name.replace(".jpg", ".txt")

            label_path = os.path.join(
                label_dir,
                label_name
            )

            with open(label_path, "w") as label_file:

                for left, top, right, bottom in annotations[img_name]:

                    x, y, w, h = convert_bbox_to_yolo(
                        left,
                        top,
                        right,
                        bottom,
                        img_w,
                        img_h
                    )

                    # ALL SIGNS = CLASS 0
                    label_file.write(
                        f"0 {x} {y} {w} {h}\n"
                    )

    # Save datasets
    save_split(train_images, "train")
    save_split(val_images, "val")

    # Create YAML
    yaml_path = os.path.join(
        output_dir,
        f"{dataset_name}.yaml"
    )

    yaml_content = f"""path: {output_dir}
train: images/train
val: images/val

names:
  0: traffic_sign
"""

    with open(yaml_path, "w") as f:
        f.write(yaml_content)

    print("Created:", dataset_name)
    print("Classes: 1")
    print("Train images:", len(train_images))
    print("Val images:", len(val_images))
    print("Total images used:", len(image_names))
    print("YAML file:", yaml_path)

In [ ]:
import os
import shutil
import random
from PIL import Image

DATASET_PATH = "/content/drive/MyDrive/research/TrainIJCNN2013"

GT_FILE = os.path.join(DATASET_PATH, "gt.txt")

OUTPUT_BASE = "/content/drive/MyDrive/research/yolo_datasets"

In [ ]:
# ==============================
# CONVERT BBOX TO YOLO FORMAT
# ==============================

def convert_bbox_to_yolo(left, top, right, bottom, img_w, img_h):

    x_center = ((left + right) / 2) / img_w
    y_center = ((top + bottom) / 2) / img_h

    width = (right - left) / img_w
    height = (bottom - top) / img_h

    return x_center, y_center, width, height

In [ ]:
# ==============================
# CREATE GENERIC DATASET
# ==============================

create_generic_traffic_sign_dataset()

Created: generic_traffic_sign
Classes: 1
Train images: 404
Val images: 102
Total images used: 506
YAML file: /content/drive/MyDrive/research/yolo_datasets/generic_traffic_sign/generic_traffic_sign.yaml
